## 2. Detection & Segmentation

In [2]:
!pip install pycocotools opencv-python matplotlib timm ultralytics segmentation-models-pytorch torchmetrics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 12.7 MB/s eta 0:00:00


In [3]:
import os
import json
from tqdm import tqdm

def extract_categories(anno_dir):

    categories = set()

    files = os.listdir(anno_dir)

    for file in tqdm(files):

        if not file.endswith(".json"):
            continue

        path = os.path.join(anno_dir, file)

        with open(path) as f:
            data = json.load(f)

        for key in data:

            if "item" not in key:
                continue

            item = data[key]

            if "category_name" in item:
                categories.add(item["category_name"])

    return sorted(list(categories))

In [4]:
os.chdir('/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd')

In [5]:
import os
import json
from collections import Counter
from tqdm import tqdm

train_anno_dir = "./train/train/train/annos"

category_counter = Counter()

files = os.listdir(train_anno_dir)

for file in tqdm(files):

    if not file.endswith(".json"):
        continue

    path = os.path.join(train_anno_dir, file)

    with open(path) as f:
        data = json.load(f)

    for key in data:

        if "item" not in key:
            continue

        category = data[key]["category_name"]

        category_counter[category] += 1

100%|██████████| 191961/191961 [26:48<00:00, 119.32it/s]


In [6]:
print("Category frequencies:\n")

for cat, count in category_counter.most_common():
    print(cat, ":", count)

Category frequencies:

short sleeve top : 71645
trousers : 55387
shorts : 36616
long sleeve top : 36064
skirt : 30835
vest dress : 17949
short sleeve dress : 17211
vest : 16095
long sleeve outwear : 13457
long sleeve dress : 7907
sling dress : 6492
sling : 1985
short sleeve outwear : 543


In [7]:
TOP5 = [cat for cat, _ in category_counter.most_common(5)]

print("Top 5 categories:")
print(TOP5)

Top 5 categories:
['short sleeve top', 'trousers', 'shorts', 'long sleeve top', 'skirt']


In [8]:
label_map = {cat:i for i,cat in enumerate(TOP5)}

print("Label mapping:")

for k,v in label_map.items():
    print(k,"->",v)

Label mapping:
short sleeve top -> 0
trousers -> 1
shorts -> 2
long sleeve top -> 3
skirt -> 4


In [15]:
import os, json, cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE   = 512
NUM_CLASSES = 5   # reuses label_map and TOP5 from Section 1

TRAIN_IMG  = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/image"
TRAIN_ANNO = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/annos"
VAL_IMG    = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/validation/validation/validation/image"
VAL_ANNO   = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/validation/validation/validation/annos"

train_df = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/train_labels_top5_new.csv")
val_df   = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/val_labels_top5_new.csv")


In [21]:
CLASS_COLS = list(label_map.keys())

def stratified_sample(df, cols, n_per_class):
    selected = set()
    for col in cols:
        idxs = df[df[col] == 1].index.tolist()
        selected.update(idxs[:n_per_class])
    return df.loc[sorted(selected)].reset_index(drop=True)

train_df_det = stratified_sample(train_df, CLASS_COLS, n_per_class=600)
val_df_det   = stratified_sample(val_df,   CLASS_COLS, n_per_class=150)

print(f"Train subset : {len(train_df_det)} images")
print(f"Val   subset : {len(val_df_det)}   images")
print("\nPer-class counts (train):")
print(train_df_det[CLASS_COLS].sum().to_string())

Train subset : 2175 images
Val   subset : 617   images

Per-class counts (train):
short sleeve top    947
trousers            689
shorts              600
long sleeve top     657
skirt               600


In [23]:
def parse_anno(img_name, anno_dir, label_map):
    path = os.path.join(anno_dir, img_name.replace(".jpg", ".json"))
    items = []
    if not os.path.exists(path):
        return items
    with open(path) as f:
        data = json.load(f)
    for key in data:
        if "item" not in key:
            continue
        item = data[key]
        cat  = item.get("category_name", "")
        if cat not in label_map:
            continue
        bbox = item.get("bounding_box", None)
        segs = item.get("segmentation", [])
        if bbox is None or not segs:
            continue
        items.append({"label": label_map[cat], "bbox": bbox, "segmentation": segs})
    return items

def poly_to_mask(segmentation, H, W):
    mask = np.zeros((H, W), dtype=np.uint8)
    for poly in segmentation:
        pts = np.array(poly, dtype=np.float32).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [pts], 1)
    return mask

def collate_fn(batch):
    return tuple(zip(*batch))


### 2.1 Mask R-CNN (Transfer Learning)

In [25]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn  import MaskRCNNPredictor

class MRCNNDataset(Dataset):
    def __init__(self, df, img_dir, anno_dir, size=IMG_SIZE):
        self.df = df; self.img_dir = img_dir
        self.anno_dir = anno_dir; self.size = size

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        W, H = img.size
        img  = img.resize((self.size, self.size))
        img_t = TF.to_tensor(img)
        sx, sy = self.size / W, self.size / H

        boxes, labels, masks = [], [], []
        for ann in parse_anno(img_name, self.anno_dir, label_map):
            x1, y1, x2, y2 = [c * s for c, s in zip(ann["bbox"], [sx, sy, sx, sy])]
            if x2 <= x1 or y2 <= y1:
                continue
            sc = []
            for poly in ann["segmentation"]:
                # fix: scale as float, then cast to int32
                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                pts[:, 0] *= sx; pts[:, 1] *= sy
                sc.append(pts.astype(np.int32).flatten().tolist())
            boxes.append([x1, y1, x2, y2])
            labels.append(ann["label"] + 1)
            masks.append(poly_to_mask(sc, self.size, self.size))

        if len(boxes) == 0:
            t = {"boxes":  torch.zeros((0, 4), dtype=torch.float32),
                 "labels": torch.zeros((0,),   dtype=torch.int64),
                 "masks":  torch.zeros((0, self.size, self.size), dtype=torch.uint8)}
        else:
            t = {"boxes":  torch.tensor(boxes,          dtype=torch.float32),
                 "labels": torch.tensor(labels,         dtype=torch.int64),
                 "masks":  torch.tensor(np.stack(masks), dtype=torch.uint8)}
        t["image_id"] = torch.tensor([idx])
        return img_t, t

train_mrcnn_dl = DataLoader(MRCNNDataset(train_df_det, TRAIN_IMG, TRAIN_ANNO),
                             batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_mrcnn_dl   = DataLoader(MRCNNDataset(val_df_det,   VAL_IMG,   VAL_ANNO),
                             batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

In [26]:
def get_maskrcnn():
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_f  = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor  = FastRCNNPredictor(in_f, NUM_CLASSES + 1)
    in_m  = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_m, 256, NUM_CLASSES + 1)
    return model

In [ ]:
maskrcnn  = get_maskrcnn().to(DEVICE)
opt_mrcnn = torch.optim.Adam([p for p in maskrcnn.parameters() if p.requires_grad], lr=1e-4)

EPOCHS = 5
best_mrcnn_loss = float("inf")

for epoch in range(EPOCHS):
    maskrcnn.train()
    total = 0
    for imgs, targets in train_mrcnn_dl:
        imgs    = [i.to(DEVICE) for i in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss    = sum(maskrcnn(imgs, targets).values())
        opt_mrcnn.zero_grad(); loss.backward(); opt_mrcnn.step()
        total  += loss.item()
    avg = total / len(train_mrcnn_dl)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.4f}")
    if avg < best_mrcnn_loss:
        best_mrcnn_loss = avg
        torch.save(maskrcnn.state_dict(), "/kaggle/working/best_maskrcnn.pth")

Epoch 1/5 | Loss: 0.6412


In [ ]:
from torchmetrics.detection import MeanAveragePrecision

maskrcnn.load_state_dict(torch.load("/kaggle/working/best_maskrcnn.pth"))
maskrcnn.eval()

metric_mrcnn  = MeanAveragePrecision(iou_type="segm")
iou_cls_mrcnn = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for imgs, targets in val_mrcnn_dl:
        imgs  = [i.to(DEVICE) for i in imgs]
        preds = maskrcnn(imgs)

        # threshold float masks → uint8 before metric update
        preds_fixed = []
        for p in preds:
            preds_fixed.append({
                "boxes":  p["boxes"].cpu(),
                "scores": p["scores"].cpu(),
                "labels": p["labels"].cpu(),
                "masks":  (p["masks"].squeeze(1).cpu() > 0.5).to(torch.uint8)
            })

        targets_fixed = [{k: v.cpu() for k, v in t.items()} for t in targets]

        metric_mrcnn.update(preds_fixed, targets_fixed)

        for pred, tgt in zip(preds_fixed, targets_fixed):
            for pm, pl in zip(pred["masks"], pred["labels"]):
                pb = pm.numpy()
                c  = pl.item() - 1
                if not (0 <= c < NUM_CLASSES): continue
                gt = tgt["masks"][tgt["labels"] == (c + 1)].numpy()
                if len(gt) == 0: continue
                iou_cls_mrcnn[c].append(
                    max((pb & g).sum() / ((pb | g).sum() + 1e-6) for g in gt))

res_mrcnn     = metric_mrcnn.compute()
per_iou_mrcnn = [np.mean(v) if v else 0.0 for v in iou_cls_mrcnn]
miou_mrcnn    = np.mean(per_iou_mrcnn)
dice_mrcnn    = np.mean([2 * iou / (1 + iou) for iou in per_iou_mrcnn])

In [ ]:
print("Mask R-CNN Results")
print(f"mAP@[0.5:0.95] (segm) : {res_mrcnn['map'].item():.4f}")
print(f"mAP@0.50       (segm) : {res_mrcnn['map_50'].item():.4f}")
print(f"mIoU                  : {miou_mrcnn:.4f}")
print(f"Dice (macro)          : {dice_mrcnn:.4f}")
print(f"Per-class IoU         : {[f'{v:.3f}' for v in per_iou_mrcnn]}")

### 2.2 YOLO (Transfer Learning — YOLOv8s-seg)

In [ ]:
import yaml
from ultralytics import YOLO

YOLO_DIR = "/kaggle/working/yolo_data"

In [ ]:
def build_yolo_split(df, img_dir, anno_dir, split):
    img_out   = f"{YOLO_DIR}/images/{split}"
    label_out = f"{YOLO_DIR}/labels/{split}"
    os.makedirs(img_out,   exist_ok=True)
    os.makedirs(label_out, exist_ok=True)

    for _, row in df.iterrows():
        name = row["image"]
        src  = os.path.join(img_dir, name)
        if not os.path.exists(src):
            continue

        # symlink image — zero extra disk usage
        stem    = os.path.splitext(name)[0]
        dst_img = f"{img_out}/{stem}.jpg"
        if not os.path.exists(dst_img):
            os.symlink(src, dst_img)

        # read image dims for normalisation only (no copy)
        img = cv2.imread(src)
        if img is None:
            continue
        H, W = img.shape[:2]

        lines = []
        for ann in parse_anno(name, anno_dir, label_map):
            cls      = ann["label"]
            pts_flat = []
            for poly in ann["segmentation"]:
                arr = np.array(poly, dtype=np.float32).reshape(-1, 2)
                arr[:, 0] /= W; arr[:, 1] /= H
                pts_flat.extend(arr.flatten().tolist())
            if pts_flat:
                lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in pts_flat))

        with open(f"{label_out}/{stem}.txt", "w") as f:
            f.write("\n".join(lines))

build_yolo_split(train_df_det, TRAIN_IMG, TRAIN_ANNO, "train")
build_yolo_split(val_df_det,   VAL_IMG,   VAL_ANNO,   "val")

yaml.dump({"path": YOLO_DIR, "train": "images/train", "val": "images/val",
           "nc": NUM_CLASSES, "names": list(label_map.keys())},
          open(f"{YOLO_DIR}/data.yaml", "w"))
print("YOLO dataset ready (images symlinked, no copy)")


In [ ]:
os.chdir("/kaggle/working")

In [ ]:
yolo_model = YOLO("yolov8s-seg.pt")
yolo_model.train(
    data     = f"{YOLO_DIR}/data.yaml",
    epochs   = 20,          
    imgsz    = 416,          # was 512, saves memory
    batch    = 8,
    device   = 0 if torch.cuda.is_available() else "cpu",
    project  = "/kaggle/working",
    name     = "yolo_seg_v2",
    exist_ok = True,
    patience = 10,           # early stopping
    overlap_mask = True,     # helps with overlapping clothing
    mask_ratio   = 2,        # finer masks
)

In [ ]:
val_res_yolo = yolo_model.val(data=f"{YOLO_DIR}/data.yaml")

map_box_yolo = val_res_yolo.box.map
map_seg_yolo = val_res_yolo.seg.map

In [ ]:
print("YOLO Results")
print(f"Box  mAP@[0.5:0.95] : {map_box_yolo:.4f}")
print(f"Seg  mAP@[0.5:0.95] : {map_seg_yolo:.4f}")
print(f"Box  mAP@0.50       : {val_res_yolo.box.map50:.4f}")
print(f"Seg  mAP@0.50       : {val_res_yolo.seg.map50:.4f}")

### 2.3 U-Net (Transfer Learning — ResNet34 encoder)

In [ ]:
import segmentation_models_pytorch as smp

class UNetDataset(Dataset):
    def __init__(self, df, img_dir, anno_dir, size=IMG_SIZE, aug=None):
        self.df = df; self.img_dir = img_dir
        self.anno_dir = anno_dir; self.size = size; self.aug = aug

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.size, self.size))

        W_orig, H_orig = Image.open(os.path.join(self.img_dir, img_name)).size
        sx, sy = self.size / W_orig, self.size / H_orig

        seg_mask = np.zeros((self.size, self.size), dtype=np.int64)
        for ann in parse_anno(img_name, self.anno_dir, label_map):
            sc = []
            for poly in ann["segmentation"]:
                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                pts[:, 0] *= sx; pts[:, 1] *= sy
                sc.append(pts.astype(np.int32).flatten().tolist())
            m = poly_to_mask(sc, self.size, self.size)
            seg_mask[m == 1] = ann["label"] + 1

        if self.aug:
            out    = self.aug(image=img, mask=seg_mask.astype(np.uint8))
            img_t  = out["image"].float()
            mask_t = torch.tensor(out["mask"], dtype=torch.long)
        else:
            img_t  = TF.to_tensor(img)
            mask_t = torch.tensor(seg_mask, dtype=torch.long)

        return img_t, mask_t

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.Affine(translate_percent=0.1, scale=(0.8, 1.2), rotate=(-15, 15), p=0.5),
    A.GridDistortion(p=0.2),
    A.CoarseDropout(num_holes_range=(4, 8), hole_height_range=(16, 32),
                    hole_width_range=(16, 32), p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

val_aug = A.Compose([
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

In [ ]:
train_df_unet = stratified_sample(train_df, CLASS_COLS, n_per_class=2000)
val_df_unet   = stratified_sample(val_df,   CLASS_COLS, n_per_class=200)

print(f"U-Net Train: {len(train_df_unet)} | Val: {len(val_df_unet)}")

train_unet_dl = DataLoader(UNetDataset(train_df_unet, TRAIN_IMG, TRAIN_ANNO, aug=train_aug),
                            batch_size=8, shuffle=True,  num_workers=2)
val_unet_dl   = DataLoader(UNetDataset(val_df_unet,   VAL_IMG,   VAL_ANNO,  aug=val_aug),
                            batch_size=8, shuffle=False, num_workers=2)

In [ ]:
unet = smp.Unet(encoder_name="resnet50", encoder_weights="imagenet",
                in_channels=3, classes=NUM_CLASSES + 1).to(DEVICE)

# class weights: downweight background (class 0), upweight clothing
weights = torch.tensor([0.2, 1.0, 1.0, 1.0, 1.0, 1.0], dtype=torch.float32).to(DEVICE)
ce_loss   = torch.nn.CrossEntropyLoss(weight=weights)
dice_loss = smp.losses.DiceLoss(mode="multiclass")

def combined_loss(pred, mask):
    return 0.5 * dice_loss(pred, mask) + 0.5 * ce_loss(pred, mask)
    
opt_unet  = torch.optim.Adam(unet.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt_unet, T_max=20)

In [ ]:
EPOCHS = 5
best_unet_loss = float("inf")

for epoch in range(EPOCHS):
    unet.train()
    train_total = 0
    for imgs, masks in train_unet_dl:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        loss = combined_loss(unet(imgs), masks)
        opt_unet.zero_grad(); loss.backward(); opt_unet.step()
        train_total += loss.item()

    # validate each epoch
    unet.eval()
    val_total = 0
    with torch.no_grad():
        for imgs, masks in val_unet_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            val_total += combined_loss(unet(imgs), masks).item()

    scheduler.step()
    t_avg = train_total / len(train_unet_dl)
    v_avg = val_total   / len(val_unet_dl)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {t_avg:.4f} | Val: {v_avg:.4f}")

    if v_avg < best_unet_loss:
        best_unet_loss = v_avg
        torch.save(unet.state_dict(), "/kaggle/working/best_unet.pth")

In [ ]:
from scipy import ndimage

unet.load_state_dict(torch.load("/kaggle/working/best_unet.pth"))
unet.eval()

iou_cls_unet  = [[] for _ in range(NUM_CLASSES)]
dice_cls_unet = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for imgs, masks in val_unet_dl:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = unet(imgs).argmax(dim=1)
        for c in range(NUM_CLASSES):
            p = (preds == c + 1).cpu().numpy()
            g = (masks == c + 1).cpu().numpy()
            inter = (p & g).sum(); union = (p | g).sum()
            if union > 0:
                iou_cls_unet[c].append(inter / (union + 1e-6))
                dice_cls_unet[c].append(2 * inter / (p.sum() + g.sum() + 1e-6))

In [ ]:
per_iou_unet  = [np.mean(v) if v else 0.0 for v in iou_cls_unet]
per_dice_unet = [np.mean(v) if v else 0.0 for v in dice_cls_unet]
miou_unet = np.mean(per_iou_unet)
dice_unet = np.mean(per_dice_unet)

print("U-Net Results")
print(f"mIoU         : {miou_unet:.4f}")
print(f"Dice (macro) : {dice_unet:.4f}")
print(f"Per-class IoU: {[f'{v:.3f}' for v in per_iou_unet]}")

In [ ]:
# Instance post-processing via connected components
def unet_instances(pred_mask):
    instances = []
    for c in range(NUM_CLASSES):
        binary = (pred_mask == c + 1).astype(np.uint8)
        labeled, n = ndimage.label(binary)
        for i in range(1, n + 1):
            comp = (labeled == i)
            ys, xs = np.where(comp)
            if len(xs) < 10: continue
            instances.append({
                "bbox":  [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
                "mask":  comp.astype(np.uint8), "label": c})
    return instances

### 2.4 Summary Table

In [ ]:
summary = pd.DataFrame({
    "Model"          : ["Mask R-CNN", "YOLOv8s-seg", "U-Net"],
    "mAP@[0.5:0.95]" : [f"{res_mrcnn['map'].item():.4f}", f"{map_seg_yolo:.4f}", "N/A"],
    "mIoU"           : [f"{miou_mrcnn:.4f}", "—", f"{miou_unet:.4f}"],
    "Dice (macro)"   : [f"{dice_mrcnn:.4f}", "—", f"{dice_unet:.4f}"],
})
print(summary.to_string(index=False))